## Using linear regression to draw causal conclusions.

**The material was prepared for the course [Explainable machine learning](https://ai-interpretability.school).**

Trying to estimate the connection between a feature and the target variable, researchers often resort to analysing the correlation coefficient. Besides, the correlation coefficient is the first thing people are introduced to on Data science courses, so even beginners know about it. However:

> *Correlation is not causation.*

or "correlation is not equal to a causal relationship". But what should we do if we still want to estimate a causal relationship?

If you are taking the course [Explainable machine learning](https://ai-interpretability.school), then in the block on linear models we have already got acquainted with using regression coefficients to analyse feature importance. Now we suggest studying the possibility of using regression coefficients at a deeper level!

To make the material clear to everyone, let us start from the very beginning.

In [ ]:
#Importing the libraries
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

Shall we estimate the importance of various features for our careless students?

**The mathematics**

Let:\
$T$ -- the feature whose influence we want to estimate (the willingness to get a higher education)\
$Y$ -- the values of the target variable without introducing the additional feature\
$Y_0$ -- the values of the target variable given that the student was not willing to go to university\
$Y_1$ -- the values of the target variable given that the student was willing to go to university\
$TE$ -- the amplitude of the change of the feature when the value changes from 0 to 1

In [ ]:
#In a perfect world our dataset would look like this
ate_att_data = pd.DataFrame(dict(
      i= [1,2,3,4],
      Y0=[500,600,800,700],
      Y1=[450,600,600,750],
      T= [0,0,1,1],
      Y= [500,600,600,750],
      TE=[-50,0,-200,50],
))
ate_att_data

In [ ]:
#But in the real one it looks like this
pd.DataFrame(dict(
    i= [1,2,3,4],
    Y0=[500,600,np.nan,np.nan],
    Y1=[np.nan,np.nan,600,750],
    T= [0,0,1,1],
    Y= [500,600,600,750],
    TE=[np.nan,np.nan,np.nan,np.nan],
))

We have a lot of students. For convenience, let us introduce a couple more things:

- $Y_{i0}$ - the potential grade of the $i-th$ student, when they are not willing to go to university\
- $Y_{i1}$ - the potential grade of the $i-th$ student, when they are willing to go to university

- Then the influence of the feature "willingness to get a higher education" can be defined as: $Y_{i1} - Y_{i0}$


- $ATE$ *average treatment effect* — the average effect of the presence of the feature\
- $ATT$ *average treatment effect on treated* — the average effect of the presence of the feature for the observations that have this feature

Note that:\
$ATE = E[Y_1 - Y_0]$ — the expectation of the difference of the target value with the presence and the absence of the feature\
$ATT = E[Y_1 - Y_0|T=1]$ -  the expectation of the difference of the target value with the presence and the absence of the feature given that the feature is equal to 1.

With the perfect dataset we would be able to compute everything by definition. With the real one — no. And since we have decided to work with reality, let us consider:

$$E[Y|T=1] - E[Y|T=0] = E[Y_1|T=1] - E[Y_0|T=0] = E[Y_1|T=1] - E[Y_0|T=1] + E[Y_0|T=1] - E[Y_0|T=0]=E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$$


We got: $E[Y_1-Y_0|T=1] + (E[Y_0|T=1] - E[Y_0|T=0])$

The first term $E[Y_1-Y_0|T=1]$ determines the importance of the feature, the second one $(E[Y_0|T=1] - E[Y_0|T=0])$ — the random noise (bias).

**Statement of the problem:** \
The influence of some variables cannot be checked using A/B testing. For example:

- Some experiments are unethical. You cannot make students consume alcohol in order to see how that would affect their performance at university.
- Running a number of other experiments is forbidden by legal restrictions. For example, you cannot set different prices for one and the same product.
- There are also situations where running an experiment is impossible. If a rebranding is happening, launching the product on two sufficiently large non-overlapping groups can be expensive.


In this case the researcher has to work with what there is.

What can help to estimate the influence of a feature? **Linear regression!**

**The dataset used:** [data on the performance of students, University of California, Irvine](https://archive.ics.uci.edu/dataset/320/student+performance).


Let us find out how the willingness to get a higher education affects the final grade in mathematics. Namely, let us try to build the model:

$$ math_{-}grade = \beta_0+k*higher + τ,$$
where:
- $\beta_0$ – the intercept
- $higher$ – the willingness to get a higher education
- $τ$ — the random noise

In [ ]:
data = pd.read_csv('https://github.com/aiedu-courses/all_datasets/raw/main/student-mat.csv',
                   sep=';')
data.head()

In [ ]:
model = smf.ols('G3 ~ higher', data=data).fit()
model.summary().tables[1]

**What does each coefficient mean?**
- coef - for Intercept, the value of the target variable for the whole sample when all the features are equal to 0; for a feature, the approximate change of the prediction if the feature is equal to 1;
- std err	-- the standard deviation of the influence (of the weight)
- pvalue — the significance level; since it is less than 0.05 here, we can say that the willingness to get a higher education significantly affects the prediction

But the information from the regression does not end there!

In [ ]:
model.summary()

Here:
- When the model was trained;
- The number of observations
- The training method, OLS
- The quality of the regression: $R^2, R^2_{adj}$
- F-statistic: the value corresponding to Fisher's test with the null hypothesis "the regression is not significant", that is, it is not informative at all
- Prob (F-statistic): the probability that the hypothesis about the non-informativeness of the regression is true
- Log-Likelihood: the log-likelihood over the sample	-1154.5
- AIC, BIC: criteria for evaluating the regression

**Conclusions. Let us emphasise that:**

- Modelling "cause-effect" relationships is limited by the presence of hidden interconnections in the data. That is why any causal conclusions based on observations should be treated with a degree of scepticism. We cannot be sure that the estimate we obtained is completely objective.

- Having observational data, we can only put forward hypotheses and test them on the basis of models;

- The regression from statsmodels stores the information that allows you to analyse the model you built straight after building it, without additional tests.

Thank you for the time you spent on this material!

Sources:
1. [Casual Inference for the Brave and True](https://matheusfacure.github.io/python-causality-handbook/01-Introduction-To-Causality.html)
2. [Linear regressions for causal conclusions](https://towardsdatascience.com/linear-regressions-for-causal-conclusions-34c6317c5a11)